# 5.3 자전거 대여 수요 예측 (Bike Sharing) - 주석 설명판

원본 `5_3.ipynb`의 코드에 설명 주석을 단 버전이에요. (앞에서 고친 오류는 모두 반영된 상태)

**전체 흐름**
1. 데이터 불러오기 → 날짜(datetime)를 연/월/일/시로 쪼개기
2. 선형회귀로 첫 예측 → 평가 (RMSLE, RMSE, MAE)
3. 타깃값 로그 변환 → 다시 예측
4. 원-핫 인코딩 → 선형회귀 / 릿지 / 라쏘 비교
5. 트리 기반 회귀 모델(랜덤 포레스트, GBM, XGBoost, LightGBM) 비교

## 1. 라이브러리 불러오기

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')   # 경고 메시지 숨기기

## 2. 데이터 불러오기 / 구조 확인

In [ ]:
# 캐글 Bike Sharing Demand 데이터: 1시간 = 1행
#  - season(계절), holiday(공휴일), workingday(근무일), weather(날씨)
#  - temp(기온), atemp(체감온도), humidity(습도), windspeed(풍속)
#  - casual(비회원 대여수), registered(회원 대여수), count(전체 대여수) ← count가 예측할 타깃
df = pd.read_csv('./bike_train.csv')
print(df.shape)   # (10886, 12)
df.head()

In [ ]:
df.info()
# → datetime 컬럼이 object(문자열) 타입 → 날짜로 바꿔야 연/월/시를 꺼낼 수 있음

## 3. 날짜 컬럼 가공

In [ ]:
# 문자열 '2011-01-01 00:00:00' → 날짜 타입(datetime64)으로 변환
df['datetime'] = df.datetime.apply(pd.to_datetime)
df.info()   # datetime 타입이 datetime64[ns]로 바뀐 걸 확인

In [ ]:
# 날짜에서 연, 월, 일, 시를 각각 새 컬럼으로 꺼냄
# lambda x: x.year → '각 값 x에 대해 x.year를 돌려줘'라는 짧은 함수
df['year'] = df.datetime.apply(lambda x: x.year)
df['month'] = df.datetime.apply(lambda x: x.month)
df['day'] = df.datetime.apply(lambda x: x.day)
df['hour'] = df.datetime.apply(lambda x: x.hour)
df.head()

In [ ]:
# 필요 없는 컬럼 삭제
#  - datetime: 이미 year/month/day/hour로 쪼갰으니 필요 없음
#  - casual, registered: 둘을 더하면 count(정답)가 됨
#    → 이걸 피처로 쓰면 정답을 미리 알려주는 셈이라 반드시 빼야 함
drop_columns = ['datetime', 'casual', 'registered']
df.drop(drop_columns, axis=1, inplace=True)

## 4. 피처별 평균 대여량 시각화

In [ ]:
# 2행 x 4열 = 그래프 8개
fig, axs = plt.subplots(figsize=(16, 8), ncols=4, nrows=2)
cat_features = ['year', 'month', 'season', 'weather', 'day', 'hour', 'holiday', 'workingday']

for i, feature in enumerate(cat_features):
    row = int(i / 4)   # 0~3번째는 0행, 4~7번째는 1행
    col = i % 4        # 4로 나눈 나머지 → 0, 1, 2, 3 반복
    # 피처 값별 count의 평균을 막대로 그림
    sns.barplot(x=feature, y='count', data=df, ax=axs[row][col])

# 해석:
#  - year: 2012년이 2011년보다 대여량이 많음
#  - hour: 출퇴근 시간(8시, 17~18시)에 대여량이 높음
#  - weather: 날씨가 나쁠수록(1→3) 대여량이 적음
#    (weather 4는 데이터가 1개뿐이라 막대가 들쭉날쭉함)

## 5. 평가 함수 만들기 (RMSLE, RMSE, MAE)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

def rmsle(y, pred):
    """RMSLE: 로그를 씌운 뒤 계산한 RMSE → '몇 % 틀렸는지'를 봄 (이 대회의 공식 평가 지표)"""
    log_y = np.log1p(y)        # log(1 + 실제값)
    log_pred = np.log1p(pred)  # log(1 + 예측값)
    squared_error = (log_y - log_pred) ** 2
    rmsle = np.sqrt(np.mean(squared_error))
    return rmsle

def rmse(y, pred):
    """RMSE: 오차 제곱 평균의 제곱근 → '평균 몇 대 틀렸는지'"""
    return np.sqrt(mean_squared_error(y, pred))

def evaluate_regr(y, pred):
    """세 가지 지표를 한 번에 출력 (셋 다 낮을수록 좋음)"""
    rmsle_val = rmsle(y, pred)
    rmse_val = rmse(y, pred)
    mae_val = mean_absolute_error(y, pred)   # MAE: 오차 절댓값의 평균
    print('RMSLE: {0:.3f}, RMSE: {1:.3f}, MAE: {2:.3f}'.format(rmsle_val, rmse_val, mae_val))

## 6. 선형회귀로 첫 예측

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso

y_target = df['count']                                   # 정답: 대여량
X_features = df.drop(['count'], axis=1, inplace=False)   # 문제: 나머지 컬럼

# 학습 70% / 테스트 30%
X_train, X_test, y_train, y_test = train_test_split(X_features, y_target,
                                                    test_size=0.3, random_state=42)

lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train)
pred = lr_reg.predict(X_test)

evaluate_regr(y_test, pred)
# 결과: RMSLE 1.174, RMSE 141.261, MAE 105.676
# → 평균 대여량이 약 190대인데 평균 140대 정도 틀림 = 성능이 많이 나쁨

## 7. 오차가 가장 큰 데이터 확인

In [ ]:
def get_top_error_data(y_test, pred, n_tops=5):
    # 실제값과 예측값을 나란히 놓은 표 만들기
    res_df = pd.DataFrame(y_test.values, columns=['real_count'])
    res_df['pred_count'] = np.round(pred)
    # 차이의 절댓값 = 얼마나 틀렸는지
    res_df['diff'] = np.abs(res_df['real_count'] - res_df['pred_count'])
    # 많이 틀린 순으로 n_tops개 출력
    print(res_df.sort_values('diff', ascending=False)[:n_tops])

get_top_error_data(y_test, pred, n_tops=5)
# → 실제 900대 넘게 빌린 시간을 300대 정도로 예측함
#   대여량이 큰 값을 제대로 못 맞힘 → 타깃 분포가 치우쳐 있는지 확인해 보자

## 8. 타깃값 분포 확인 → 로그 변환

In [ ]:
# 로그 변환 전: 0 근처에 몰려 있고 오른쪽으로 꼬리가 긴 모양 (치우친 분포)
y_target.hist()

In [ ]:
# 로그 변환 후: 정규분포에 더 가까운 모양
# np.log1p(x) = log(1 + x) → 큰 값은 많이, 작은 값은 조금 줄여줌
y_log_transform = np.log1p(y_target)
y_log_transform.hist()

## 9. 로그 변환한 타깃으로 다시 학습

In [ ]:
# 타깃을 로그 변환
y_target_log = np.log1p(y_target)

X_train, X_test, y_train, y_test = train_test_split(X_features, y_target_log,
                                                    test_size=0.3, random_state=42)

lr_reg = LinearRegression()
lr_reg.fit(X_train, y_train)
pred = lr_reg.predict(X_test)   # 예측값도 '로그 대여량'으로 나옴

# 평가는 원래 단위(대여 대수)로 해야 하니까 다시 되돌림
# np.expm1(x) = e^x - 1 → np.log1p의 반대 함수
y_test_exp = np.expm1(y_test)   # 정답을 원래 크기로
pred_exp = np.expm1(pred)       # 예측을 원래 크기로

evaluate_regr(y_test_exp, pred_exp)
# 결과: RMSLE 1.020 (좋아짐), RMSE 160.730 (오히려 나빠짐)
# → 아직 부족함. 다음 그래프에서 원인을 찾아보자

## 10. 회귀 계수 확인

In [ ]:
# 피처별 회귀 계수를 큰 순으로 정렬
coef = pd.Series(lr_reg.coef_, index=X_features.columns)
coef_sort = coef.sort_values(ascending=False)

# x=계수 값, y=피처 이름 → 가로 막대그래프
sns.barplot(x=coef_sort.values, y=coef_sort.index)

# 해석: year 계수가 압도적으로 큼
#  - year 값은 2011, 2012라서 숫자 자체가 아주 큼
#  - month, hour, season, weather 같은 값도 사실은 '종류(범주)'인데
#    모델은 숫자 크기로 받아들임 (예: hour 23이 hour 1보다 23배 중요하다고 착각)
#  → 이런 범주형 피처는 원-핫 인코딩을 해야 함

## 11. 원-핫 인코딩

In [ ]:
# 범주형 피처를 0/1 컬럼으로 펼침
#  예) hour → hour_0, hour_1, ..., hour_23 (해당 시간만 1, 나머지는 0)
#  → 숫자 크기에 의미가 없어지고 '어떤 종류인가'만 남음
X_features_ohe = pd.get_dummies(X_features,
                                columns=['year', 'month', 'day', 'hour', 'holiday',
                                         'workingday', 'season', 'weather'])

## 12. 선형회귀 / 릿지 / 라쏘 비교

In [ ]:
# 원-핫 인코딩된 피처 + 로그 변환된 타깃으로 다시 나누기
X_train, X_test, y_train, y_test = train_test_split(X_features_ohe, y_target_log,
                                                    test_size=0.3, random_state=42)

def get_model_predict(model, X_train, X_test, y_train, y_test, is_expm1=False):
    """모델 학습 → 예측 → 평가를 한 번에"""
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    # 타깃을 로그 변환했으면 정답과 예측을 원래 크기로 되돌린 뒤 평가
    if is_expm1:
        y_test = np.expm1(y_test)
        pred = np.expm1(pred)
    print('###', model.__class__.__name__, '###')   # 모델 이름 출력
    evaluate_regr(y_test, pred)

lr_reg = LinearRegression()
ridge_reg = Ridge(alpha=10)      # L2 규제: 계수를 전체적으로 줄임
lasso_reg = Lasso(alpha=0.01)    # L1 규제: 중요하지 않은 계수를 0으로 만듦

for model in [lr_reg, ridge_reg, lasso_reg]:
    get_model_predict(model, X_train, X_test, y_train, y_test, is_expm1=True)

# 결과:
#   LinearRegression  RMSLE 0.585, RMSE 95.845
#   Ridge             RMSLE 0.587, RMSE 96.827
#   Lasso             RMSLE 0.639, RMSE 112.044
# → 원-핫 인코딩 전(RMSLE 1.020)보다 크게 좋아짐!

## 13. 원-핫 인코딩 후 회귀 계수 상위 20개

In [ ]:
coef = pd.Series(lr_reg.coef_, index=X_features_ohe.columns)
coef_sort = coef.sort_values(ascending=False)[:20]   # 큰 순으로 20개만
sns.barplot(x=coef_sort.values, y=coef_sort.index)

# ⚠ 주의: 계수 값이 1e10(100억) 단위로 말도 안 되게 크게 나옴
#  - 원-핫 컬럼끼리 서로 완벽하게 겹치는 정보가 있어서 그래요 (다중공선성)
#    예) month_7~9이면 무조건 season_3 / year_2011 + year_2012 = 항상 1
#  - 이러면 계수끼리 +100억, -100억처럼 서로 상쇄하면서 예측은 맞추게 됨
#    → 예측 성능은 괜찮지만 '계수 크기 = 중요도'로 해석하면 안 됨
#  - 규제가 있는 릿지(ridge_reg)의 계수로 그려 보면 정상적인 크기로 나옴
#    → 상위: hour_17, hour_18, hour_8 (출퇴근 시간이 대여량을 가장 크게 올림)
#  아래 줄 주석을 풀고 실행해서 비교해 보세요
# coef = pd.Series(ridge_reg.coef_, index=X_features_ohe.columns)
# coef_sort = coef.sort_values(ascending=False)[:20]
# sns.barplot(x=coef_sort.values, y=coef_sort.index)

## 14. 트리 기반 회귀 모델 비교

In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# n_estimators=500 → 트리를 500개 만들어서 결과를 합침
rf = RandomForestRegressor(n_estimators=500)        # 여러 트리의 평균 (배깅)
gbm = GradientBoostingRegressor(n_estimators=500)   # 앞 트리의 오차를 다음 트리가 보완 (부스팅)
xgb = XGBRegressor(n_estimators=500)                # GBM을 빠르고 강하게 개선한 버전
lgbm = LGBMRegressor(n_estimators=500)              # XGBoost보다 더 빠른 부스팅

for model in [rf, gbm, xgb, lgbm]:
    # .values: DataFrame → numpy 배열 (XGBoost/LightGBM 버전에 따라 컬럼 이름 문제를 피하려고)
    get_model_predict(model, X_train.values, X_test.values,
                      y_train.values, y_test.values, is_expm1=True)

# 결과 (random_state를 안 정해서 소수점 아래가 조금씩 달라질 수 있음):
#   RandomForest  RMSLE 0.349, RMSE 48.1  (이 모델이 실행마다 조금씩 달라짐)
#   GBM           RMSLE 0.331, RMSE 51.7
#   XGBoost       RMSLE 0.344, RMSE 50.5
#   LightGBM      RMSLE 0.314, RMSE 44.5  ← 가장 좋음
# → 선형 모델(RMSLE 약 0.59)보다 훨씬 좋음
#   트리 모델은 '17시 + 근무일이면 많다' 같은 복잡한 조건 조합을 잘 잡아내기 때문

## 정리

| 단계 | 모델 | RMSLE |
|---|---|---|
| 처음 | 선형회귀 | 1.174 |
| 타깃 로그 변환 | 선형회귀 | 1.020 |
| + 원-핫 인코딩 | 선형회귀 | 0.585 |
| 트리 기반 모델 | LightGBM | 0.314 |

**배운 점**
- 치우친 타깃값은 **로그 변환** (`np.log1p`) → 평가할 때 **`np.expm1`** 으로 되돌리기
- 숫자로 되어 있어도 '종류'를 뜻하는 피처(시간, 월, 계절 등)는 **원-핫 인코딩**
- 이 데이터처럼 조건 조합이 중요하면 **트리 기반 모델**이 선형 모델보다 강함